In [1]:
import os
import re
from getpass import getpass

# 1. CONFIGURACIÓN DE LA API DE GROQ
print("=== CONFIGURACIÓN DEL ENTORNO ===")
if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass("Introduce tu GROQ API Key (obtenla en console.groq.com): ")
print("✅ API Key configurada correctamente.\n")

# 2. ESCUDO DE SEGURIDAD (PII) - Heredado de tu v4.1
def analizar_seguridad(texto):
    """
    Detecta patrones PII, los anonimiza y devuelve el texto limpio 
    junto con el reporte de hallazgos.
    """
    patrones = {
        "DNI": r"\b\d{8}[A-Z]\b",
        "Teléfono": r"\b[6789]\d{8}\b",
        "Matrícula": r"\b\d{4}[A-Z]{3}\b",
        "IBAN": r"\bES\d{22}\b"
        # Quitamos la póliza de aquí porque la validaremos en su propio estado
    }
    
    hallazgos = []
    texto_limpio = texto
    
    for tipo, regex in patrones.items():
        matches = re.findall(regex, texto_limpio, re.IGNORECASE)
        for m in matches:
            anonimo = m[:2] + "*" * (len(m)-4) + m[-2:]
            hallazgos.append({"tipo": tipo, "anonimo": anonimo})
            texto_limpio = texto_limpio.replace(m, f"[{tipo} PROTEGIDO]")
            
    return hallazgos, texto_limpio

=== CONFIGURACIÓN DEL ENTORNO ===


Introduce tu GROQ API Key (obtenla en console.groq.com):  ········


✅ API Key configurada correctamente.



In [2]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("=== INICIALIZANDO CONOCIMIENTO RAG (FAISS) ===")

# 1. Definición de la Base de Conocimiento con sus Metadatos e IDs Únicos
documentos_segurplus = [
    # ==================== SEGUROS DE AUTOMÓVIL ====================
    Document(
        page_content="Introducción: Los seguros de automóvil de SegurPlus protegen al conductor frente a daños propios, daños a terceros, robo y asistencia en carretera.",
        metadata={"id": "AUT_INTRO_001", "titulo": "Introducción", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "general", "categoria": "general", "subcategoria": "introduccion", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO BÁSICO (TERCEROS AMPLIADO) - Responsabilidad Civil Obligatoria: Cobertura exigida por la legislación española. Incluye daños materiales y personales causados a terceros.",
        metadata={"id": "AUT_BAS_001", "titulo": "Responsabilidad Civil Obligatoria", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_basico", "categoria": "coberturas", "subcategoria": "responsabilidad_civil_obligatoria", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO BÁSICO - Responsabilidad Civil Voluntaria y Defensa Jurídica: Amplía los límites legales de responsabilidad civil hasta 50.000.000 €. Incluye Defensa Jurídica con defensa penal y reclamación de daños con un límite de 6.000 €.",
        metadata={"id": "AUT_BAS_002", "titulo": "Responsabilidad Civil Voluntaria", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_basico", "categoria": "coberturas", "subcategoria": "responsabilidad_civil_voluntaria", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO BÁSICO - Asistencia en Carretera: Disponible desde el kilómetro 0, las 24 horas, los 365 días del año. Incluye: remolque, cambio de rueda y envío de combustible. El tiempo medio estimado de llegada es de 45 minutos en zonas urbanas.",
        metadata={"id": "AUT_BAS_003", "titulo": "Asistencia en Carretera", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_basico", "categoria": "servicios", "subcategoria": "asistencia_carretera", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO BÁSICO - Robo del Vehículo: Cobertura frente a robo total e intento de robo con daños. Indemnización: Valor a nuevo durante los primeros 24 meses; valor venal mejorado posteriormente.",
        metadata={"id": "AUT_BAS_004", "titulo": "Robo", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_basico", "categoria": "coberturas", "subcategoria": "robo_vehiculo", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO BÁSICO - Incendio: Cobertura por incendio accidental, explosión y caída de rayo.",
        metadata={"id": "AUT_BAS_005", "titulo": "Incendio", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_basico", "categoria": "coberturas", "subcategoria": "incendio", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO BÁSICO - Rotura de Lunas: Incluye parabrisas, luneta trasera y ventanillas laterales. Sin franquicia.",
        metadata={"id": "AUT_BAS_006", "titulo": "Rotura de Lunas", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_basico", "categoria": "coberturas", "subcategoria": "rotura_lunas", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO BÁSICO - Exclusiones Principales: No quedan cubiertos la conducción bajo efectos de alcohol o drogas, participación en competiciones, daños provocados intencionadamente y vehículos sin ITV vigente cuando el siniestro esté relacionado.",
        metadata={"id": "AUT_BAS_007", "titulo": "Exclusiones", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_basico", "categoria": "exclusiones", "subcategoria": "general", "nivel": "basico", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO PREMIUM (TODO RIESGO) - Daños Propios: Cobertura de daños sufridos por el vehículo incluso cuando el conductor es responsable del accidente. Incluye colisiones, vuelcos y actos vandálicos.",
        metadata={"id": "AUT_PREM_001", "titulo": "Daños Propios", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_premium", "categoria": "coberturas", "subcategoria": "danos_propios", "nivel": "premium", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO PREMIUM - Vehículo de Sustitución: Disponible cuando la reparación supera las 24 horas o existe robo del vehículo. Duración máxima: 15 días.",
        metadata={"id": "AUT_PREM_002", "titulo": "Vehículo Sustitución", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_premium", "categoria": "servicios", "subcategoria": "vehiculo_sustitucion", "nivel": "premium", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO PREMIUM - Accidentes del Conductor: Capital asegurado de 50.000 € por fallecimiento y 50.000 € por invalidez permanente.",
        metadata={"id": "AUT_PREM_003", "titulo": "Accidentes del Conductor", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_premium", "categoria": "coberturas", "subcategoria": "accidentes_conductor", "nivel": "premium", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO PREMIUM - Objetos Personales: Cobertura de equipaje, ordenadores y dispositivos electrónicos dentro del coche con un límite de 2.000 €.",
        metadata={"id": "AUT_PREM_004", "titulo": "Objetos Personales", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_premium", "categoria": "coberturas", "subcategoria": "objetos_personales", "nivel": "premium", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO PREMIUM - Asistencia Premium: Incluye vehículo de sustitución inmediato, hotel en desplazamientos y transporte alternativo.",
        metadata={"id": "AUT_PREM_005", "titulo": "Asistencia Premium", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_premium", "categoria": "servicios", "subcategoria": "asistencia_premium", "nivel": "premium", "documento": "catalogo_auto", "version": "1.0"}
    ),
    Document(
        page_content="AUTO PREMIUM - Daños por Fenómenos Naturales y Franquicias: Cobertura por granizo, inundaciones y tempestades cuando no sean asumidos por el Consorcio. Franquicias disponibles a elección del cliente: 150 €, 300 € o 600 €.",
        metadata={"id": "AUT_PREM_006", "titulo": "Fenómenos Naturales", "compania": "SegurPlus", "tipo_seguro": "auto", "producto": "auto_premium", "categoria": "coberturas", "subcategoria": "fenomenos_naturales", "nivel": "premium", "documento": "catalogo_auto", "version": "1.0"}
    ),

    # ==================== SEGUROS DE HOGAR ====================
    Document(
        page_content="Introducción Hogar: SegurPlus ofrece soluciones de protección para viviendas. Los seguros de hogar cubren daños materiales, responsabilidad civil y asistencia urgente en el domicilio.",
        metadata={"id": "HOG_INTRO_001", "titulo": "Introducción Hogar", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "general", "subcategoria": "introduccion", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR ESENCIAL - Incendio y explosión: Se cubren los daños ocasionados por incendios accidentales, explosiones de gas doméstico y caída de rayo.",
        metadata={"id": "HOG_ESE_001", "titulo": "Incendio y Explosión", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "coberturas", "subcategoria": "incendio_explosion", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR ESENCIAL - Daños por agua: Se cubren daños provocados por rotura accidental de tuberías, fugas en instalaciones fijas y desbordamiento de depósitos domésticos. Límite máximo: 30.000 € por siniestro. No cubre falta de mantenimiento.",
        metadata={"id": "HOG_ESE_002", "titulo": "Daños por Agua", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "coberturas", "subcategoria": "danos_agua", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR ESENCIAL - Fenómenos atmosféricos: Se cubren daños causados por lluvia intensa, viento superior a 80 km/h y granizo. Límite máximo de 20.000 €.",
        metadata={"id": "HOG_ESE_003", "titulo": "Fenómenos Atmosféricos", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "coberturas", "subcategoria": "fenomenos_atmosfericos", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR ESENCIAL - Robo en la vivienda: Se cubre robo con signos de fuerza y daños en puertas o ventanas. Capital máximo asegurado para contenido: 10.000 €.",
        metadata={"id": "HOG_ESE_004", "titulo": "Robo", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "coberturas", "subcategoria": "robo_vivienda", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR ESENCIAL - Responsabilidad Civil Familiar: Cobertura por daños involuntarios causados a terceros (ej. fuga de agua que afecta al vecino, caída de objetos). Límite: 150.000 €.",
        metadata={"id": "HOG_ESE_005", "titulo": "Responsabilidad Civil", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "coberturas", "subcategoria": "responsabilidad_civil", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR ESENCIAL - Asistencia Hogar 24 horas: Incluye fontanero, electricista y cerrajero urgente. Tiempo máximo de respuesta: 4 horas en capitales de provincia.",
        metadata={"id": "HOG_ESE_006", "titulo": "Asistencia Hogar", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "servicios", "subcategoria": "asistencia_24h", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR ESENCIAL - Exclusiones Principales: Falta de mantenimiento, humedades por condensación, inundaciones del Consorcio, actos intencionados y viviendas deshabitadas más de 90 días.",
        metadata={"id": "HOG_ESE_007", "titulo": "Exclusiones", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_esencial", "categoria": "exclusiones", "subcategoria": "general", "nivel": "basico", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR PREMIUM - Daños estéticos: Cuando una reparación provoca diferencias visuales, cubre la reposición estética (ej. sustitución completa de azulejos). Límite: 15.000 €.",
        metadata={"id": "HOG_PREM_001", "titulo": "Daños Estéticos", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_premium", "categoria": "coberturas", "subcategoria": "danos_esteticos", "nivel": "premium", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR PREMIUM - Robo fuera del hogar: Cobertura de bienes robados fuera de la vivienda (ej. robo de bolso, ordenador portátil). Límite: 3.000 € por siniestro.",
        metadata={"id": "HOG_PREM_002", "titulo": "Robo fuera del hogar", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_premium", "categoria": "coberturas", "subcategoria": "robo_fuera_hogar", "nivel": "premium", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR PREMIUM - Equipos electrónicos: Cobertura de televisores, ordenadores, tablets y consolas. Incluye daños por sobretensión eléctrica con un límite de 8.000 €.",
        metadata={"id": "HOG_PREM_003", "titulo": "Equipos Electrónicos", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_premium", "categoria": "coberturas", "subcategoria": "equipos_electronicos", "nivel": "premium", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR PREMIUM - Asistencia Informática: Incluye eliminación de virus, configuración de dispositivos y recuperación de datos. Hasta 3 intervenciones anuales.",
        metadata={"id": "HOG_PREM_004", "titulo": "Asistencia Informática", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_premium", "categoria": "servicios", "subcategoria": "asistencia_informatica", "nivel": "premium", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR PREMIUM - Defensa Jurídica: Reclamación de daños, defensa en conflictos vecinales y asistencia legal telefónica con un límite de 12.000 €.",
        metadata={"id": "HOG_PREM_005", "titulo": "Defensa Jurídica", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_premium", "categoria": "servicios", "subcategoria": "defensa_juridica", "nivel": "premium", "documento": "catalogo_hogar", "version": "1.0"}
    ),
    Document(
        page_content="SEGURO HOGAR PREMIUM - Servicio de Manitas: Incluye montaje de muebles, instalación de cortinas y colocación de estanterías. Hasta 3 servicios al año.",
        metadata={"id": "HOG_PREM_006", "titulo": "Reparaciones", "compania": "SegurPlus", "tipo_seguro": "hogar", "producto": "hogar_premium", "categoria": "servicios", "subcategoria": "reparaciones", "nivel": "premium", "documento": "catalogo_hogar", "version": "1.0"}
    )
]

# 2. Inicializar el modelo de embeddings (gratuito y local)
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 3. Construir el almacén de vectores FAISS indexando los documentos
vector_store = FAISS.from_documents(documentos_segurplus, embeddings_model)

print("✅ Base de datos FAISS creada con éxito en memoria.")

=== INICIALIZANDO CONOCIMIENTO RAG (FAISS) ===


2026-06-04 11:54:06.293467: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/anaconda3/envs/IA_MUBA/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Base de datos FAISS creada con éxito en memoria.


In [3]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

class SegurPlusLLM:
    """Chatbot con LLM (Groq), memoria evolutiva y RAG con FAISS."""
    
    def __init__(self, tipo_incidencia, num_poliza, vector_store):
        self.llm = ChatGroq(
            model_name="llama-3.3-70b-versatile",
            temperature=0.2
        )
        self.history = []
        self.max_history = 5 
        
        self.tipo_incidencia = tipo_incidencia
        self.num_poliza = num_poliza
        self.vector_store = vector_store
        
    def respond(self, user_input: str) -> str:
        # 1. Determinar el filtro meta para FAISS (Coche -> auto, Hogar -> hogar)
        tipo_meta = "auto" if "coche" in self.tipo_incidencia.lower() else "hogar"
        
        # 2. RAG: Recuperar fragmentos del catálogo
        docs_relevantes = self.vector_store.similarity_search(
            user_input, 
            k=3,
            filter={"tipo_seguro": tipo_meta}
        )
        contexto_rag = "\n\n".join([doc.page_content for doc in docs_relevantes])
        
        # 3. Detectar si es modo Informativo o modo Incidencia Activa
        es_modo_info = "información" in self.tipo_incidencia.lower()
        
        if es_modo_info:
            instrucciones_objetivo = """TU OBJETIVO ES MERAMENTE INFORMATIVO:
            - Responde a las dudas del usuario utilizando estrictamente la INFORMACIÓN OFICIAL provista.
            - Explica con claridad qué cubre cada producto (Básico vs Premium), sus límites económicos y exclusiones.
            - Como es una consulta informativa, NO ofrezcas enviar servicios de emergencia a menos que el cliente te pregunte explícitamente cómo funcionaría dicho trámite."""
        else:
            instrucciones_objetivo = f"""TU OBJETIVO ES RESOLVER UNA INCIDENCIA ACTIVA:
            - El número de póliza validado del cliente es: {self.num_poliza}.
            - Determina la solución de asistencia adecuada basada en el relato del cliente y el catálogo de coberturas.
            - Tus opciones de resolución inmediata son: Mandar un técnico a casa, Mandar una grúa, Llamar a una ambulancia, Pedir un taxi, o Pasar con un agente humano (para quejas o dudas complejas).
            - Cuando identifiques el problema, confírmale al cliente la acción exacta que vas a ejecutar."""

        # 4. Construir el Prompt del Sistema unificado
        system_prompt = f"""Eres un agente de asistencia inteligente de SegurPlus.
        Estás en el departamento de: {self.tipo_incidencia.upper()}.
        
        === INFORMACIÓN OFICIAL DE LAS PÓLIZAS SEGURPLUS ===
        {contexto_rag}
        ====================================================
        
        {instrucciones_objetivo}
        
        REGLAS CRÍTICAS:
        1. Sé muy empático, profesional y directo.
        2. Bájate en los datos reales del catálogo. Si algo no viene o está excluido, indícalo educadamente.
        3. Haz solo una pregunta a la vez si necesitas recopilar más datos.
        4. NUNCA pidas DNI, teléfonos o IBAN en la conversación.
        """
        
        # 5. Compilar mensajes y ejecutar
        messages = [SystemMessage(content=system_prompt)]
        for human_msg, ai_msg in self.history[-self.max_history:]:
            messages.append(HumanMessage(content=human_msg))
            messages.append(AIMessage(content=ai_msg))
            
        messages.append(HumanMessage(content=user_input))
        
        response = self.llm.invoke(messages)
        ai_response = response.content
        
        self.history.append((user_input, ai_response))
        return ai_response

In [ ]:
import re

def iniciar_chatbot_segurplus():
    print("\n" + "="*50)
    print("🤖 SISTEMA SEGURPLUS v6.0 (RAG & RUTA INTELIGENTE)")
    print("="*50)
    
    # Variables de estado del sistema
    estado_actual = "MENU_PRINCIPAL"
    tipo_incidencia = None
    num_poliza = None
    bot_llm = None
    
    # Mensaje de bienvenida con la nueva estructura solicitada
    print("\nChatbot: ¡Hola! Bienvenido a SegurPlus. Por favor, selecciona el motivo de tu consulta:")
    print("  1. Quiero información de los seguros")
    print("  2. He tenido una incidencia / Necesito asistencia urgente")
    print("  3. Modificación de Datos / Gestiones Administrativas")

    while True:
        usuario = input("\nTú: ").strip()
        
        if usuario.lower() in ['salir', 'exit', 'quit']:
            print("Chatbot: Gracias por confiar en SegurPlus. ¡Que tengas un excelente día!")
            break
            
        if not usuario:
            continue

        # PASO A: Anonimización de datos (Protección PII activa en todo momento)
        alerta_datos, texto_seguro = analizar_seguridad(usuario)
        if alerta_datos:
            print("\n🛡️ [REPORTE DE PRIVACIDAD]")
            for dato in alerta_datos:
                print(f"  - He ocultado el dato tipo {dato['tipo']} ({dato['anonimo']}) por tu seguridad.")

        # =========================================================================
        # ESTADO 0: MENÚ RAÍZ (Nueva segmentación de Intenciones)
        # =========================================================================
        if estado_actual == "MENU_PRINCIPAL":
            if "1" in texto_seguro or "información" in texto_seguro.lower() or "informacion" in texto_seguro.lower():
                estado_actual = "INFO_TIPO"
                print("\nChatbot: Perfecto. ¿Sobre qué ramo de seguros deseas obtener información?")
                print("  1. Seguro de Coche")
                print("  2. Seguro de Hogar")
                
            elif "2" in texto_seguro or "incidencia" in texto_seguro.lower() or "asistencia" in texto_seguro.lower():
                estado_actual = "CONTEXTO"
                print("\nChatbot: Lamento escuchar eso. Vamos a gestionar tu asistencia de inmediato. ¿De qué seguro se trata?")
                print("  1. Seguro de Coche")
                print("  2. Seguro de Hogar")
                
            elif "3" in texto_seguro or "datos" in texto_seguro.lower() or "gestiones" in texto_seguro.lower():
                tipo_incidencia = "Gestiones Administrativas"
                estado_actual = "POLIZA"
                print("\nChatbot: Entendido, área de Gestiones Administrativas. Por favor, indícame tu número de póliza (8 dígitos).")
            else:
                print("\nChatbot: Por favor, introduce una opción válida (1, 2 o 3) para poder derivarte.")

        # =========================================================================
        # ESTADO 1.1: SELECCIÓN DE RAMO PARA CONSULTAS INFORMATIVAS (RAG directo sin póliza)
        # =========================================================================
        elif estado_actual == "INFO_TIPO":
            if "1" in texto_seguro or "coche" in texto_seguro.lower():
                tipo_incidencia = "Información Coche"
                num_poliza = "NO REQUERIDA"
                bot_llm = SegurPlusLLM(tipo_incidencia, num_poliza, vector_store)
                estado_actual = "LLM_CHAT"
                print("\nChatbot: ¡Excelente! Tengo el catálogo de Automóviles cargado (Básico y Premium). Pregúntame lo que quieras sobre coberturas, límites, talleres o exclusiones.")
            elif "2" in texto_seguro or "hogar" in texto_seguro.lower():
                tipo_incidencia = "Información Hogar"
                num_poliza = "NO REQUERIDA"
                bot_llm = SegurPlusLLM(tipo_incidencia, num_poliza, vector_store)
                estado_actual = "LLM_CHAT"
                print("\nChatbot: ¡Excelente! Tengo el catálogo de Hogar listo (Esencial y Premium). Puedes consultarme sobre daños por agua, robos, servicio de manitas, coberturas, etc.")
            else:
                print("\nChatbot: Por favor, selecciona 1 (Coche) o 2 (Hogar) para cargar el catálogo correcto.")

        # =========================================================================
        # ESTADO 1.2: SELECCIÓN DE RAMO PARA INCIDENCIAS (Flujo clásico operativo)
        # =========================================================================
        elif estado_actual == "CONTEXTO":
            if "1" in texto_seguro or "coche" in texto_seguro.lower():
                tipo_incidencia = "Coche"
                estado_actual = "POLIZA"
                print("\nChatbot: Entendido, apertura de incidencia para Coche. Por favor, indícame tu número de póliza (8 dígitos).")
            elif "2" in texto_seguro or "hogar" in texto_seguro.lower():
                tipo_incidencia = "Hogar"
                estado_actual = "POLIZA"
                print("\nChatbot: Entendido, apertura de incidencia para Hogar. Por favor, indícame tu número de póliza (8 dígitos).")
            else:
                print("\nChatbot: Por favor, responde con 1 o 2 para identificar el tipo de siniestro.")
                
        # =========================================================================
        # ESTADO 2: VALIDACIÓN REQUERIDA DE PÓLIZA (Sólo para Incidencias y Gestiones)
        # =========================================================================
        elif estado_actual == "POLIZA":
            match_poliza = re.search(r"\b\d{8}\b", texto_seguro)
            if match_poliza:
                num_poliza = match_poliza.group(0)
                estado_actual = "LLM_CHAT"
                # Instanciamos el bot operativo pasándole la base de datos vectorial
                bot_llm = SegurPlusLLM(tipo_incidencia, num_poliza, vector_store)
                print(f"\nChatbot: Póliza {num_poliza} verificada con éxito. Cuéntame con detalle qué ha ocurrido para tramitar la asistencia.")
            else:
                print("\nChatbot: Estructura incorrecta. Recuerda que para proceder necesitamos los 8 dígitos numéricos de tu póliza.")
                
        # =========================================================================
        # ESTADO 3: CONVERSACIÓN FLUIDA CON LLM + CONTEXTO RAG INYECTADO
        # =========================================================================
        elif estado_actual == "LLM_CHAT":
            respuesta_llm = bot_llm.respond(texto_seguro)
            print(f"\nChatbot: {respuesta_llm}")

# Iniciar la ejecución de la aplicación actualizada
iniciar_chatbot_segurplus()


🤖 SISTEMA SEGURPLUS v6.0 (RAG & RUTA INTELIGENTE)

Chatbot: ¡Hola! Bienvenido a SegurPlus. Por favor, selecciona el motivo de tu consulta:
  1. Quiero información de los seguros
  2. He tenido una incidencia / Necesito asistencia urgente
  3. Modificación de Datos / Gestiones Administrativas
